## PropertyLens RAG v2 — full multi-source pipeline

**What's new vs v1:**
- All four data sources embedded into **separate Pinecone namespaces** (`transactions`, `amenities`, `xai`, `trends`)
- **Free-text queries** — no metadata required from the user; Gemma 3 extracts town/flat_type/year via NLP
- **Namespace router** — each query targets only the relevant namespace(s)
- **Prediction tool** — extracts structured fields from the query and calls the joblib `HybridClusterEnsemble` model directly
- Gemma 3 receives both RAG context **and** the model prediction in one prompt

### Prerequisites
- `.env` at repo root with `PINECONE_API_KEY=...`
- Ollama running locally: `ollama pull gemma3`
- `data/artifacts/model_bundle.pkl` (or adjust `MODEL_BUNDLE_PATH` in config)


In [1]:
# ── all dependencies in one cell ──────────────────────────────────────────────
# pinecone + pinecone-text  → hybrid vector index + BM25 sparse encoder
# sentence-transformers     → BGE-M3 dense embeddings (local)
# transformers + torch      → cross-encoder reranker (BAAI/bge-reranker-v2-m3)
# ollama                    → Gemma 3 local LLM
# joblib                    → load the HybridClusterEnsemble model bundle
# pandas / numpy / tqdm     → data processing + progress bars
# python-dotenv             → load .env secrets
!pip install -q pinecone pinecone-text sentence-transformers transformers torch \
               ollama joblib pandas numpy tqdm python-dotenv

### Configuration

All secrets and tuneable knobs live here. Set `PINECONE_API_KEY` in a `.env` file at the repo root.
Adjust `MODEL_BUNDLE_PATH` to point at your actual joblib artefact.


In [2]:
from __future__ import annotations
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ── Pinecone ───────────────────────────────────────────────────────────────────
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX   = "propertylens-rag"

# One namespace per data source — keeps retrieval scoped and fast
NS_TRANSACTIONS = "transactions"
NS_AMENITIES    = "amenities"
NS_XAI          = "xai"
NS_TRENDS       = "trends"

# ── Embedding / reranking models ───────────────────────────────────────────────
DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DIMENSION  = 1024
RERANKER_MODEL   = "BAAI/bge-reranker-v2-m3"

# ── Retrieval knobs ────────────────────────────────────────────────────────────
TOP_K_RETRIEVAL = 50    # candidates per namespace from hybrid search
TOP_K_RERANK    = 10    # after cross-encoder
TOP_K_MMR       = 5     # after MMR diversity filter
TOP_K_FINAL     = 5     # sent to LLM
MMR_LAMBDA      = 0.7   # 1.0 = pure relevance, 0.0 = pure diversity
RRF_K           = 60    # standard RRF constant
N_SUBQUERIES    = 3     # multi-query reformulations

# ── LLM ───────────────────────────────────────────────────────────────────────
OLLAMA_MODEL    = "gemma3"
OLLAMA_BASE_URL = "http://localhost:11434"


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repo root by searching upward for `data/` and `notebooks/`."""
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError(
        "Could not locate repo root. Expected to find `data/` and `notebooks/` directories above current working directory."
    )


REPO_ROOT = find_repo_root()

# ── Data paths ─────────────────────────────────────────────────────────────────
DATA_ROOT          = str(REPO_ROOT / "data")
TRANSACTIONS_GLOB  = str(Path(DATA_ROOT) / "feature_data" / "**" / "outputs" / "*.csv")
AMENITIES_GLOB     = str(Path(DATA_ROOT) / "amenities" / "*.csv")
ARTIFACTS_DIR      = str(Path(DATA_ROOT) / "artifacts")
#MODEL_BUNDLE_PATH  = str(Path(ARTIFACTS_DIR) / "model_bundle.pkl")  # adjust if needed
MODEL_BUNDLE_PATH = str(Path(ARTIFACTS_DIR) / "hybrid_cluster_bundle.joblib")

print("Config loaded.")
print(f"Pinecone index : {PINECONE_INDEX}")
print(f"REPO_ROOT      : {REPO_ROOT}")
print(f"DATA_ROOT      : {DATA_ROOT}")
print(f"Model bundle   : {MODEL_BUNDLE_PATH}")

Config loaded.
Pinecone index : propertylens-rag
REPO_ROOT      : /Users/bhuvesh/Documents/PropertyLens
DATA_ROOT      : /Users/bhuvesh/Documents/PropertyLens/data
Model bundle   : /Users/bhuvesh/Documents/PropertyLens/data/artifacts/hybrid_cluster_bundle.joblib


### Load raw data

Loads all four sources into DataFrames.
- `transactions_df` — HDB resale records (the main data)
- `amenities_df` — MRT, schools, malls, hawker centres per town
- `trends_df` — derived: median price per town per year
- `xai_bundle` — dict loaded from the model bundle artefact (SHAP, CBR, rules)


In [3]:
from __future__ import annotations
import glob
import pickle
import joblib
import pandas as pd


def _infer_from_onehots(df: pd.DataFrame, prefix: str) -> pd.Series | None:
    """Infer a categorical value from one-hot columns like town_* or flat_type_*."""
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        return None
    return df[cols].idxmax(axis=1).str.replace(prefix, "", regex=False)


def load_transactions(pattern: str) -> pd.DataFrame:
    """Load all transaction CSVs matching glob pattern into one DataFrame."""
    files = glob.glob(pattern, recursive=True)
    if not files:
        raise FileNotFoundError(f"No CSVs found at: {pattern}")
    dfs = [pd.read_csv(f) for f in files]
    df = pd.concat(dfs, ignore_index=True)

    if "town" not in df.columns:
        town = _infer_from_onehots(df, "town_")
        if town is not None:
            df["town"] = town

    if "flat_type" not in df.columns:
        flat_type = _infer_from_onehots(df, "flat_type_")
        if flat_type is not None:
            df["flat_type"] = flat_type

    if "town" in df.columns:
        df["town"] = df["town"].astype(str).str.upper().str.strip()
    if "flat_type" in df.columns:
        df["flat_type"] = df["flat_type"].astype(str).str.upper().str.strip()

    print(f"  Transactions: {len(df):,} rows from {len(files)} file(s)")
    return df


def load_amenities(pattern: str) -> pd.DataFrame:
    """Load all amenity CSVs into one DataFrame, tagging source filename."""
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"No CSVs found at: {pattern}")
    dfs = []
    for f in files:
        tmp = pd.read_csv(f)
        tmp["source_file"] = os.path.basename(f)
        dfs.append(tmp)
    df = pd.concat(dfs, ignore_index=True)
    print(f"  Amenities: {len(df):,} rows from {len(files)} file(s)")
    return df


def derive_trends(transactions: pd.DataFrame) -> pd.DataFrame:
    """Compute median resale_price per town per year from transaction data."""
    if "town" not in transactions.columns:
        raise KeyError("transactions_df is missing 'town' after load_transactions()")

    year_col = "transaction_year" if "transaction_year" in transactions.columns else "year"
    if year_col not in transactions.columns and "month" in transactions.columns:
        transactions = transactions.copy()
        transactions["transaction_year"] = pd.to_datetime(
            transactions["month"], errors="coerce"
        ).dt.year
        year_col = "transaction_year"
    grp = (
        transactions.groupby(["town", year_col])["resale_price"]
        .agg(median_resale_price="median", transaction_count="count")
        .reset_index()
        .rename(columns={year_col: "year"})
    )
    print(f"  Trends: {len(grp):,} town-year rows")
    return grp


def load_xai_bundle(artifacts_dir: str) -> dict:
    """
    Load XAI artefacts from the artifacts directory.
    1) Tries monolithic pickles: model_bundle.pkl, xai_bundle.pkl
    2) Else merges the hybrid_xai/ tree (global_shap_cache.json, rules.json,
       cbr_training_data.parquet) from 04_hybrid_xai_train.ipynb.

    Returns:
        dict with keys expected by build_xai_chunks: global_shap, rules, cbr_data, ...
    """
    import json

    bundle: dict = {}
    candidates = [
        os.path.join(artifacts_dir, "model_bundle.pkl"),
        os.path.join(artifacts_dir, "xai_bundle.pkl"),
    ]
    for path in candidates:
        if os.path.exists(path):
            try:
                loaded = joblib.load(path)
                if isinstance(loaded, dict):
                    bundle.update(loaded)
                    print(f"  XAI bundle loaded from {os.path.basename(path)} — keys: {list(loaded.keys())[:8]}")
                    break
            except Exception as e:
                print(f"  WARNING: could not load {path}: {e}")

    xai_dir = os.path.join(artifacts_dir, "hybrid_xai")
    if os.path.isdir(xai_dir):
        loaded_any = False
        shap_path = os.path.join(xai_dir, "global_shap_cache.json")
        if os.path.exists(shap_path):
            try:
                with open(shap_path, encoding="utf-8") as f:
                    bundle["global_shap"] = json.load(f)
                loaded_any = True
            except Exception as e:
                print(f"  WARNING: could not load {shap_path}: {e}")

        rules_path = os.path.join(xai_dir, "rules.json")
        if os.path.exists(rules_path):
            try:
                with open(rules_path, encoding="utf-8") as f:
                    raw_rules = json.load(f)
                parts = []
                if isinstance(raw_rules, dict):
                    parts.extend(raw_rules.get("apriori") or [])
                    parts.extend(raw_rules.get("surrogate") or [])
                elif isinstance(raw_rules, list):
                    parts = raw_rules
                if parts:
                    bundle["rules"] = parts
                    loaded_any = True
            except Exception as e:
                print(f"  WARNING: could not load {rules_path}: {e}")

        cbr_path = os.path.join(xai_dir, "cbr_training_data.parquet")
        if os.path.exists(cbr_path):
            try:
                bundle["cbr_data"] = pd.read_parquet(cbr_path)
                loaded_any = True
            except Exception as e:
                print(f"  WARNING: could not load {cbr_path}: {e}")

        if loaded_any:
            print(f"  XAI artefacts merged from hybrid_xai/ — keys: {list(bundle.keys())[:12]}")

    if not bundle:
        print("  WARNING: no XAI bundle found — XAI chunks will be skipped.")
    return bundle


# ── Run ────────────────────────────────────────────────────────────────────────
print("Loading data sources...")
transactions_df = load_transactions(TRANSACTIONS_GLOB)
amenities_df    = load_amenities(AMENITIES_GLOB)
trends_df       = derive_trends(transactions_df)
xai_bundle      = load_xai_bundle(ARTIFACTS_DIR)
print("Done.")

Loading data sources...
  Transactions: 1,568,804 rows from 9 file(s)
  Amenities: 615 rows from 4 file(s)
  Trends: 312 town-year rows
  XAI artefacts merged from hybrid_xai/ — keys: ['global_shap', 'rules', 'cbr_data']
Done.


### Build transaction chunks (with town-level amenity summary)

Each HDB transaction becomes a chunk with:
- **child text** (~256 tokens) — core facts: town, type, storey, floor area, price, PSF
- **parent text** (~1024 tokens) — child + town-level amenity summary (MRT, schools, malls, hawker)

The town-level join avoids needing GPS coordinates — we match on the `town` field which is always present.
Both `text` (child) and `parent_text` are stored; only `text` is embedded and BM25-fitted.
`parent_text` is stored in Pinecone metadata and sent to the LLM at generation time.


In [4]:
from __future__ import annotations
import math
import re
from typing import Any
import numpy as np
import pandas as pd


# ── helpers ───────────────────────────────────────────────────────────────────

def _safe_float(x: Any) -> float | None:
    """Safely convert x to float; return None if invalid."""
    try:
        v = float(x)
        return v if np.isfinite(v) else None
    except Exception:
        return None


def _bucket_storey(storey_range: Any) -> str:
    """Convert storey range string to a coarse band label."""
    if storey_range is None or (isinstance(storey_range, float) and math.isnan(storey_range)):
        return "unknown"
    m = re.search(r"(\d{1,2})\s*TO\s*(\d{1,2})", str(storey_range).upper())
    if m:
        mid = (int(m.group(1)) + int(m.group(2))) / 2
    else:
        m2 = re.search(r"(\d{1,2})", str(storey_range))
        mid = float(m2.group(1)) if m2 else None
    if mid is None:
        return "unknown"
    if mid <= 5: return "01-05"
    if mid <= 12: return "06-12"
    if mid <= 20: return "13-20"
    return "21+"


def _bucket_price(price: float) -> str:
    """Bucket price into 100k bands."""
    if not np.isfinite(price) or price <= 0:
        return "unknown"
    lo = int(price // 100_000) * 100
    return f"{lo}k-{lo + 100}k"


# ── town-level amenity summary ─────────────────────────────────────────────────

HDB_TOWNS = (
    "ANG MO KIO", "BEDOK", "BISHAN", "BUKIT BATOK", "BUKIT MERAH",
    "BUKIT PANJANG", "BUKIT TIMAH", "CENTRAL AREA", "CHOA CHU KANG",
    "CLEMENTI", "GEYLANG", "HOUGANG", "JURONG EAST", "JURONG WEST",
    "KALLANG/WHAMPOA", "MARINE PARADE", "PASIR RIS", "PUNGGOL",
    "QUEENSTOWN", "SEMBAWANG", "SENGKANG", "SERANGOON", "TAMPINES",
    "TOA PAYOH", "WOODLANDS", "YISHUN",
)


def _infer_town(text: str) -> str:
    """Best-effort town extraction from a free-text address."""
    u = str(text).upper()
    for t in sorted(HDB_TOWNS, key=len, reverse=True):
        if t in u:
            return t
    return ""


def _ensure_town_column(df: pd.DataFrame) -> pd.DataFrame:
    """Add 'town' column to amenities if missing, inferred from 'address'."""
    out = df.copy()
    if "town" in out.columns and out["town"].notna().any():
        out["town"] = out["town"].str.upper().str.strip()
        return out
    if "address" in out.columns:
        out["town"] = out["address"].map(_infer_town)
    else:
        out["town"] = ""
    return out


def build_town_amenity_summary(amenities: pd.DataFrame) -> dict[str, str]:
    """
    Build a mapping of UPPERCASE town name → compact amenity summary string.

    Args:
        amenities: amenities DataFrame.

    Returns:
        dict[town, summary_string]
    """
    amenities = _ensure_town_column(amenities)
    summary: dict[str, str] = {}
    for town, grp in amenities.groupby("town"):
        if not str(town).strip():
            continue
        lines: list[str] = []
        if "source_file" in grp.columns:
            for src, sg in grp.groupby("source_file"):
                atype = str(src).replace(".csv", "").replace("_", " ").title()
                names = sg["name"].dropna().astype(str).tolist()
                ex = ", ".join(names[:3])
                suffix = ", ..." if len(names) > 3 else ""
                lines.append(f"{atype} ({len(names)}): {ex}{suffix}")
        else:
            names = grp["name"].dropna().astype(str).tolist()
            lines.append(f"Amenities ({len(names)}): {', '.join(names[:5])}")
        summary[str(town).upper().strip()] = "\n".join(lines)
    return summary


# ── chunk builders ─────────────────────────────────────────────────────────────

def _format_child(row: pd.Series) -> str:
    """Format core transaction facts into a short child chunk (~256 tokens)."""
    town      = str(row.get("town", "")).upper().strip()
    flat_type = str(row.get("flat_type", "")).upper()
    year      = int(row.get("transaction_year", 0) or 0)
    price     = _safe_float(row.get("resale_price")) or 0.0
    area      = _safe_float(row.get("floor_area_sqm"))
    psf       = (price / area / 10.7639) if area and area > 0 else None
    storey    = _bucket_storey(row.get("storey_range"))
    ctx       = f"This is an HDB resale transaction in {town}, year {year}."
    facts = [
        f"Town: {town}",
        f"Flat type: {flat_type}",
        f"Storey band: {storey}",
        f"Floor area (sqm): {area:.1f}" if area else "Floor area (sqm): unknown",
        f"Resale price (SGD): {int(round(price))}",
        f"Approx PSF (SGD): {psf:.1f}" if psf else "Approx PSF (SGD): unknown",
    ]
    return ctx + "\n" + "\n".join(facts)


def _format_parent(row: pd.Series, amenity_summary: str) -> str:
    """Format child chunk + town amenity summary into a rich parent chunk (~1024 tokens)."""
    base  = _format_child(row)
    lines = [base, "", "Nearby amenities (town-level):"]
    lines.append(amenity_summary if amenity_summary else "- No amenity data for this town.")
    return "\n".join(lines)


def build_transaction_chunks(
    transactions: pd.DataFrame,
    town_amenity_summary: dict[str, str],
) -> list[dict]:
    """
    Build Pinecone-ready chunk dicts for all HDB transactions.

    Args:
        transactions: HDB transactions DataFrame.
        town_amenity_summary: mapping from build_town_amenity_summary().

    Returns:
        List of chunk dicts with id, text, parent_text, metadata.
    """
    out: list[dict] = []
    for i, row in transactions.reset_index(drop=True).iterrows():
        town      = str(row.get("town", "")).upper().strip()
        flat_type = str(row.get("flat_type", "")).upper()
        year      = int(row.get("transaction_year", 0) or 0)
        price     = _safe_float(row.get("resale_price")) or 0.0
        area      = _safe_float(row.get("floor_area_sqm"))
        psf       = (price / area / 10.7639) if area and area > 0 else None
        addr      = str(row.get("address_key", f"row_{i}")).upper().replace(" ", "_")
        chunk_id  = f"txn_{addr}_{year}"

        out.append({
            "id":          chunk_id,
            "text":        _format_child(row),
            "parent_text": _format_parent(row, town_amenity_summary.get(town, "")),
            "metadata": {
                "source":      "transaction",
                "town":        town,
                "flat_type":   flat_type,
                "storey_band": _bucket_storey(row.get("storey_range")),
                "sale_year":   year,
                "price_band":  _bucket_price(price),
                "resale_price": int(round(price)),
                "psf":         float(round(psf, 1)) if psf is not None else None,
            },
        })
    return out


# ── run ────────────────────────────────────────────────────────────────────────
town_amenity_summary = build_town_amenity_summary(amenities_df)
txn_chunks = build_transaction_chunks(transactions_df, town_amenity_summary)
print(f"Transaction chunks: {len(txn_chunks):,}")
print("\nExample child:\n", txn_chunks[0]["text"])
print("\nExample parent (first 400 chars):\n", txn_chunks[0]["parent_text"][:400])

Transaction chunks: 1,568,804

Example child:
 This is an HDB resale transaction in ANG MO KIO, year 2023.
Town: ANG MO KIO
Flat type: 2 ROOM
Storey band: unknown
Floor area (sqm): 44.0
Resale price (SGD): 267000
Approx PSF (SGD): 563.8

Example parent (first 400 chars):
 This is an HDB resale transaction in ANG MO KIO, year 2023.
Town: ANG MO KIO
Flat type: 2 ROOM
Storey band: unknown
Floor area (sqm): 44.0
Resale price (SGD): 267000
Approx PSF (SGD): 563.8

Nearby amenities (town-level):
Hawker Centres (7): MARKET & HAWKER CENTRE (BLK 409 ANG MO KIO AVE 10), MARKET & HAWKER CENTRE (BLK 724 ANG MO KIO AVE 6), CHENG SAN MARKET AND COOKED FOOD CENTRE, ...
Malls (2):


### Build standalone amenity chunks

In v1, amenities only appeared embedded inside transaction parent_text.
Here we **also** embed them as standalone chunks in the `amenities` namespace.
This means a query like *"what amenities are near Tampines?"* can retrieve amenity
chunks directly, without needing a transaction to carry them.

One chunk = one amenity type (e.g. MRT stations) per town.


In [5]:
from __future__ import annotations
import pandas as pd


def build_amenity_chunks(amenities: pd.DataFrame) -> list[dict]:
    """
    Build one chunk per (town, amenity_type) pair.

    Each chunk text is a human-readable sentence listing the amenities
    of that type in that town, making it directly retrievable by name.

    Args:
        amenities: amenities DataFrame.

    Returns:
        List of chunk dicts.
    """
    amenities = _ensure_town_column(amenities)
    out: list[dict] = []

    group_cols = ["town", "source_file"] if "source_file" in amenities.columns else ["town"]

    for keys, grp in amenities.groupby(group_cols):
        town = str(keys[0] if isinstance(keys, tuple) else keys).upper().strip()
        if not town:
            continue
        src  = str(keys[1] if isinstance(keys, tuple) and len(keys) > 1 else "amenities").replace(".csv", "")
        atype = src.replace("_", " ").title()
        names = grp["name"].dropna().astype(str).tolist()
        names_str = ", ".join(names)

        text = (
            f"{atype} in {town} ({len(names)} total): {names_str}. "
            f"These are the {atype.lower()} amenities in the {town} HDB town."
        )
        chunk_id = f"amenity_{town}_{src}".replace(" ", "_").lower()

        out.append({
            "id":          chunk_id,
            "text":        text,
            "parent_text": text,
            "metadata": {
                "source":       "amenity",
                "town":         town,
                "amenity_type": atype,
                "count":        len(names),
            },
        })
    return out


amenity_chunks = build_amenity_chunks(amenities_df)
print(f"Amenity chunks: {len(amenity_chunks):,}")
print("\nExample:\n", amenity_chunks[0]["text"][:300])

Amenity chunks: 85

Example:
 Hawker Centres in ANG MO KIO (7 total): MARKET & HAWKER CENTRE (BLK 409 ANG MO KIO AVE 10), MARKET & HAWKER CENTRE (BLK 724 ANG MO KIO AVE 6), CHENG SAN MARKET AND COOKED FOOD CENTRE, CHONG BOON MARKET AND FOOD CENTRE, MAYFLOWER MARKET AND FOOD CENTRE, KEBUN BARU FOOD CENTRE, KEBUN BARU MARKET AND F


### Build trend chunks

One chunk per town-year combination. These answer questions like
*"Are Tampines prices rising?"* without scanning all transactions.
Stored in the `trends` namespace.


In [6]:
from __future__ import annotations
import pandas as pd


def build_trend_chunks(trends: pd.DataFrame) -> list[dict]:
    """
    Convert the trends DataFrame into one chunk per town-year pair.

    Args:
        trends: output of derive_trends().

    Returns:
        List of chunk dicts.
    """
    out: list[dict] = []
    for _, row in trends.iterrows():
        town = str(row.get("town", "")).upper().strip()
        year = int(row.get("year", 0) or 0)
        med  = float(row.get("median_resale_price", 0) or 0)
        n    = int(row.get("transaction_count", 0) or 0)
        text = (
            f"HDB resale trend for {town}, year {year}: "
            f"median resale price SGD {int(round(med)):,} across {n:,} transactions."
        )
        out.append({
            "id":          f"trend_{town}_{year}",
            "text":        text,
            "parent_text": text,
            "metadata": {
                "source":              "trend",
                "town":                town,
                "sale_year":           year,
                "resale_price":        int(round(med)),
                "transaction_count":   n,
            },
        })
    return out


trend_chunks = build_trend_chunks(trends_df)
print(f"Trend chunks: {len(trend_chunks):,}")
print("\nExample:\n", trend_chunks[0]["text"])

Trend chunks: 312

Example:
 HDB resale trend for ANG MO KIO, year 2015: median resale price SGD 355,000 across 5,390 transactions.


### Build XAI chunks

Converts artefacts from the model bundle into retrievable chunks stored in the `xai` namespace.
Three sub-sources:
- **SHAP global importances** — top features that drive price predictions overall
- **Surrogate / Apriori rules** — human-readable decision rules from the rules artefact
- **CBR comparable cases** — representative historical cases used for comparison

If an artefact key is missing from your bundle the function skips it gracefully.


In [7]:
from __future__ import annotations
import json
import pandas as pd


def _shap_chunks(bundle: dict) -> list[dict]:
    """
    Build SHAP global importance chunks.
    Looks for keys: 'global_shap', 'shap_global', 'feature_importance'.
    """
    out: list[dict] = []
    shap_data = bundle.get("global_shap") or bundle.get("shap_global") or bundle.get("feature_importance")
    if shap_data is None:
        print("  SHAP: no global shap key found — skipping.")
        return out

    # Accept dict {feature: importance} or list of (feature, value) pairs
    if isinstance(shap_data, dict):
        items = sorted(shap_data.items(), key=lambda x: abs(float(x[1])), reverse=True)[:20]
    elif isinstance(shap_data, list):
        items = shap_data[:20]
    else:
        print("  SHAP: unexpected format — skipping.")
        return out

    lines = [f"  {feat}: {val:.4f}" for feat, val in items]
    text  = "Global SHAP feature importances for HDB price prediction (top features):\n" + "\n".join(lines)
    out.append({
        "id": "xai_shap_global",
        "text": text,
        "parent_text": text,
        "metadata": {"source": "xai", "xai_type": "shap_global"},
    })
    return out


def _rule_chunks(bundle: dict) -> list[dict]:
    """
    Build rule chunks from surrogate or apriori rules.
    Looks for keys: 'rules', 'apriori_rules', 'surrogate_rules'.
    """
    out: list[dict] = []
    rules = bundle.get("rules") or bundle.get("apriori_rules") or bundle.get("surrogate_rules")
    if not rules:
        print("  Rules: no rules key found — skipping.")
        return out

    if isinstance(rules, list):
        for i, rule in enumerate(rules[:50]):
            text = f"HDB pricing rule #{i+1}: {str(rule)}"
            out.append({
                "id": f"xai_rule_{i}",
                "text": text,
                "parent_text": text,
                "metadata": {"source": "xai", "xai_type": "rule", "rule_id": i},
            })
    elif isinstance(rules, dict):
        for key, val in list(rules.items())[:50]:
            text = f"HDB pricing rule [{key}]: {str(val)}"
            out.append({
                "id": f"xai_rule_{key}",
                "text": text,
                "parent_text": text,
                "metadata": {"source": "xai", "xai_type": "rule"},
            })
    return out


def _cbr_chunks(bundle: dict) -> list[dict]:
    """
    Build CBR comparable case chunks.
    Looks for keys: 'cbr_data', 'cbr_cases', 'comparables'.
    Each case becomes a chunk with its feature values.
    """
    out: list[dict] = []
    # Avoid `a or b` — DataFrame truth value is ambiguous in boolean context.
    cbr = bundle.get("cbr_data")
    if cbr is None:
        cbr = bundle.get("cbr_cases")
    if cbr is None:
        cbr = bundle.get("comparables")
    if cbr is None:
        print("  CBR: no cbr key found — skipping.")
        return out

    if isinstance(cbr, pd.DataFrame):
        rows = cbr.head(200).to_dict(orient="records")
    elif isinstance(cbr, list):
        rows = cbr[:200]
    else:
        print("  CBR: unexpected format — skipping.")
        return out

    for i, case in enumerate(rows):
        text = "CBR comparable case: " + ", ".join(f"{k}={v}" for k, v in list(case.items())[:12])
        out.append({
            "id": f"xai_cbr_{i}",
            "text": text,
            "parent_text": text,
            "metadata": {
                "source":   "xai",
                "xai_type": "cbr",
                "town":     str(case.get("town", "")).upper(),
            },
        })
    return out


def build_xai_chunks(bundle: dict) -> list[dict]:
    """
    Build all XAI chunks from the loaded model bundle.

    Combines SHAP global importances, rules, and CBR cases.
    Skips gracefully for any missing artefact type.

    Args:
        bundle: dict loaded by load_xai_bundle().

    Returns:
        Combined list of XAI chunk dicts.
    """
    shap   = _shap_chunks(bundle)
    rules  = _rule_chunks(bundle)
    cbr    = _cbr_chunks(bundle)
    total  = shap + rules + cbr
    print(f"  SHAP chunks: {len(shap)} | Rule chunks: {len(rules)} | CBR chunks: {len(cbr)}")
    return total


xai_chunks = build_xai_chunks(xai_bundle)
print(f"XAI chunks total: {len(xai_chunks):,}")

  SHAP chunks: 1 | Rule chunks: 50 | CBR chunks: 200
XAI chunks total: 251


### Initialise embedding models

Two encoders — both applied to the `text` (child) field at ingest and query time:
1. **BGE-M3** dense encoder (1024-dim) — semantic similarity via cosine
2. **BM25Encoder** from `pinecone-text` — keyword matching (must be fitted on the full corpus)

BM25 is fitted on the union of ALL chunk texts across all four sources and cached to disk.


In [8]:
from __future__ import annotations
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder


def load_dense_encoder(model_name: str) -> SentenceTransformer:
    """Load the BGE-M3 dense encoder."""
    return SentenceTransformer(model_name)


def fit_bm25_encoder(corpus_texts: list[str], cache_path: str = "bm25_encoder_v2.pkl") -> BM25Encoder:
    """
    Fit a BM25Encoder on the full corpus and cache to disk.

    Args:
        corpus_texts: all chunk 'text' fields across every namespace.
        cache_path: pickle path for caching the fitted encoder.

    Returns:
        Fitted BM25Encoder.
    """
    p = Path(cache_path)
    if p.exists():
        print(f"  BM25: loading from cache {cache_path}")
        with p.open("rb") as f:
            return pickle.load(f)
    print(f"  BM25: fitting on {len(corpus_texts):,} documents...")
    enc = BM25Encoder()
    enc.fit(corpus_texts)
    with p.open("wb") as f:
        pickle.dump(enc, f)
    print(f"  BM25: fitted and cached to {cache_path}")
    return enc


dense_encoder = load_dense_encoder(DENSE_MODEL_NAME)

all_chunks   = txn_chunks + amenity_chunks + trend_chunks + xai_chunks
corpus_texts = [c["text"] for c in all_chunks]
bm25_encoder = fit_bm25_encoder(corpus_texts)

print(f"Dense encoder: {DENSE_MODEL_NAME}")
print(f"BM25 corpus size: {len(corpus_texts):,} documents")

/Users/bhuvesh/Documents/PropertyLens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 48050.77it/s]


  BM25: loading from cache bm25_encoder_v2.pkl
Dense encoder: BAAI/bge-m3
BM25 corpus size: 1,569,452 documents


### Initialise Pinecone and create index

Creates a single Pinecone serverless index with `dotproduct` metric (required for hybrid search).
The four data sources are separated by **namespaces** within the same index — cleaner than four
separate indexes and easier to manage.


In [9]:
from __future__ import annotations
from pinecone import Pinecone, ServerlessSpec


def init_pinecone(api_key: str) -> Pinecone:
    """Initialise and return a Pinecone client."""
    return Pinecone(api_key=api_key)


def get_or_create_index(pc: Pinecone, index_name: str, dimension: int) -> object:
    """
    Get or create a Pinecone hybrid index.

    Uses dotproduct metric (required for hybrid search with sparse vectors).

    Args:
        pc: Pinecone client.
        index_name: name of the index.
        dimension: dense vector dimension (1024 for BGE-M3).

    Returns:
        Pinecone Index object.
    """
    existing = [idx.name for idx in pc.list_indexes()]
    if index_name not in existing:
        print(f"  Creating index '{index_name}'...")
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric="dotproduct",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
    else:
        print(f"  Index '{index_name}' already exists.")
    return pc.Index(index_name)


pc    = init_pinecone(PINECONE_API_KEY)
index = get_or_create_index(pc, PINECONE_INDEX, DENSE_DIMENSION)
print(index.describe_index_stats())

  Index 'propertylens-rag' already exists.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '279',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 04:32:18 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-latency-ms': '12',
                                    'x-pinecone-response-duration-ms': '13'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'amenities': {'vector_count': 85},
                'transactions': {'vector_count': 4760},
                'trends': {'vector_count': 312},
                'xai': {'vector_count': 251}},
 'storageFullness': 0.0,
 'total_vector_count': 

### Encode and upsert all chunks into namespaced Pinecone index

Each chunk is encoded with both BGE-M3 (dense) and BM25 (sparse), then upserted into its
appropriate namespace: `transactions`, `amenities`, `xai`, or `trends`.

`parent_text` is stored inside Pinecone metadata so it can be retrieved and sent to the LLM.
Upsert happens in batches of 100 to avoid payload size limits.

**Note:** on the full dataset this cell is slow. Use `SAMPLE_FOR_TESTING = True` for a quick smoke test.


In [10]:
from __future__ import annotations
import numpy as np
from tqdm import tqdm

SAMPLE_FOR_TESTING = True   # set False to ingest all data
SAMPLE_SIZE        = 1000   # rows of transactions to use when sampling


def _encode_one(
    chunk: dict,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
) -> dict:
    """
    Encode one chunk into a Pinecone-ready record.

    Args:
        chunk: dict with id, text, parent_text, metadata.

    Returns:
        Pinecone upsert record dict.
    """
    dense  = dense_encoder.encode(chunk["text"], normalize_embeddings=True).tolist()
    sparse = bm25_encoder.encode_documents([chunk["text"]])[0]
    meta   = dict(chunk["metadata"])
    meta["parent_text"] = chunk["parent_text"][:3000]  # Pinecone metadata limit
    return {"id": chunk["id"], "values": dense, "sparse_values": sparse, "metadata": meta}


def upsert_namespace(
    index,
    chunks: list[dict],
    namespace: str,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    batch_size: int = 100,
) -> None:
    """
    Encode and upsert a list of chunks into the given Pinecone namespace.

    Args:
        index: Pinecone index object.
        chunks: list of chunk dicts.
        namespace: target Pinecone namespace string.
        dense_encoder: fitted BGE-M3 encoder.
        bm25_encoder: fitted BM25Encoder.
        batch_size: records per upsert call.
    """
    records = []
    for chunk in tqdm(chunks, desc=f"Encoding {namespace}"):
        records.append(_encode_one(chunk, dense_encoder, bm25_encoder))

    for i in range(0, len(records), batch_size):
        batch = records[i : i + batch_size]
        index.upsert(vectors=batch, namespace=namespace)

    print(f"  Upserted {len(records):,} records → namespace '{namespace}'")


# ── optionally sample transactions for quick testing ──────────────────────────
if SAMPLE_FOR_TESTING:
    import random
    sample = random.sample(txn_chunks, min(SAMPLE_SIZE, len(txn_chunks)))
    print(f"SAMPLE MODE: using {len(sample):,} transaction chunks (set SAMPLE_FOR_TESTING=False for full ingest)")
else:
    sample = txn_chunks
    print(f"FULL MODE: {len(sample):,} transaction chunks")

# ── upsert each namespace ─────────────────────────────────────────────────────
upsert_namespace(index, sample,         NS_TRANSACTIONS, dense_encoder, bm25_encoder)
upsert_namespace(index, amenity_chunks, NS_AMENITIES,    dense_encoder, bm25_encoder)
upsert_namespace(index, trend_chunks,   NS_TRENDS,       dense_encoder, bm25_encoder)
upsert_namespace(index, xai_chunks,     NS_XAI,          dense_encoder, bm25_encoder)

print("\nAll namespaces upserted.")
print(index.describe_index_stats())

SAMPLE MODE: using 1,000 transaction chunks (set SAMPLE_FOR_TESTING=False for full ingest)


Encoding transactions: 100%|██████████| 1000/1000 [00:22<00:00, 44.51it/s]


  Upserted 1,000 records → namespace 'transactions'


Encoding amenities: 100%|██████████| 85/85 [00:03<00:00, 23.89it/s]


  Upserted 85 records → namespace 'amenities'


Encoding trends: 100%|██████████| 312/312 [00:05<00:00, 53.61it/s]


  Upserted 312 records → namespace 'trends'


Encoding xai: 100%|██████████| 251/251 [00:07<00:00, 34.16it/s]


  Upserted 251 records → namespace 'xai'

All namespaces upserted.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '279',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 04:33:15 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-latency-ms': '2',
                                    'x-pinecone-response-duration-ms': '4'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'amenities': {'vector_count': 85},
                'transactions': {'vector_count': 5647},
                'trends': {'vector_count': 312},
                'xai': {'vector_count': 251}},
 'storageFullness': 0.0,
 

### NLP filter extractor — free-text query → structured Pinecone filters

Real users type natural language like *"Is $580k fair for a 4-room in Tampines?"*.
They do not provide `{"town": "TAMPINES", "flat_type": "4 ROOM"}` metadata dicts.

This cell uses **Gemma 3** to extract structured fields from the query.
If extraction is uncertain the function returns `None` (no filter) — Pinecone then
searches all records in the namespace, which is safe but less precise.

Also includes a **namespace router** that decides which namespaces to query
based on keywords in the question.


In [11]:
from __future__ import annotations
import json
import re
import ollama


def _set_ollama_host(base_url: str) -> None:
    """Configure Ollama client host."""
    import os
    os.environ["OLLAMA_HOST"] = base_url


def extract_filters_from_query(query: str, model: str = OLLAMA_MODEL) -> dict | None:
    """
    Use Gemma 3 to extract Pinecone metadata filters from a free-text query.

    Extracts: town, flat_type, sale_year (all optional).
    Returns None if no reliable fields can be extracted.

    Args:
        query: raw user question.
        model: Ollama model name.

    Returns:
        dict of filter fields, or None.
    """
    _set_ollama_host(OLLAMA_BASE_URL)

    prompt = f"""Extract structured fields from this Singapore HDB property query.
Return ONLY a valid JSON object with these optional keys:
  - "town": one of the Singapore HDB towns in ALL CAPS (e.g. "TAMPINES", "BEDOK")
  - "flat_type": one of "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE"
  - "sale_year": integer year if mentioned
If a field is not clearly stated, omit it. Return {{}} if nothing is clear.
Return ONLY JSON, no explanation.

Query: {query}
"""
    try:
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
        )
        raw = (response.get("message") or {}).get("content", "{}")
        # Strip markdown fences if present
        raw = re.sub(r"```[\w]*", "", raw).strip()
        parsed = json.loads(raw)
        return parsed if parsed else None
    except Exception as e:
        print(f"  Filter extraction failed: {e} — proceeding without filter.")
        return None


def route_namespaces(query: str) -> list[str]:
    """
    Decide which Pinecone namespaces to query based on keywords in the query.

    Rules (non-exclusive — multiple namespaces can be selected):
    - Always includes 'transactions' (price comparables always useful)
    - 'amenities'  if query mentions MRT, school, mall, hawker, near, amenity
    - 'trends'     if query mentions trend, rising, falling, price history, recent, year
    - 'xai'        if query mentions explain, SHAP, feature, why, reason, driver

    Args:
        query: raw user question.

    Returns:
        List of namespace strings to query.
    """
    q = query.lower()
    ns = [NS_TRANSACTIONS]  # always include comparables

    amenity_kw = ["mrt", "school", "mall", "hawker", "near", "amenity", "transport", "bus"]
    trend_kw   = ["trend", "rising", "falling", "increase", "decrease", "history",
                  "recent", "last year", "past", "over time", "appreciation"]
    xai_kw     = ["explain", "shap", "feature", "why", "reason", "driver", "factor",
                  "importan", "predict", "model say"]

    if any(kw in q for kw in amenity_kw):
        ns.append(NS_AMENITIES)
    if any(kw in q for kw in trend_kw):
        ns.append(NS_TRENDS)
    if any(kw in q for kw in xai_kw):
        ns.append(NS_XAI)

    return ns


# ── smoke test ────────────────────────────────────────────────────────────────
test_query = "Is $580k fair for a 4-room in Tampines?"
filters    = extract_filters_from_query(test_query)
namespaces = route_namespaces(test_query)
print(f"Query   : {test_query}")
print(f"Filters : {filters}")
print(f"Namespaces: {namespaces}")

Query   : Is $580k fair for a 4-room in Tampines?
Filters : {'town': 'TAMPINES', 'flat_type': '4 ROOM'}
Namespaces: ['transactions']


### Hybrid retrieval functions (dense + sparse per namespace)

We call Pinecone twice per namespace — once with `alpha=1.0` (pure dense / cosine) and once
with `alpha=0.0` (pure sparse / BM25) — to get two independent ranked lists for RRF.
Results from all namespace calls are then merged by RRF into one ranked list.


In [12]:
from __future__ import annotations
from typing import Any, Optional
import numpy as np


def _scale_sparse(sparse: dict, scale: float) -> dict:
    """Scale sparse vector values by a scalar (for alpha blending)."""
    return {"indices": sparse["indices"], "values": [v * scale for v in sparse["values"]]}


def _hybrid_query(
    index,
    query: str,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    alpha: float,
    top_k: int,
    namespace: str,
    metadata_filter: Optional[dict] = None,
) -> list[dict[str, Any]]:
    """
    Run one Pinecone hybrid query.

    Args:
        alpha: 1.0 = pure dense, 0.0 = pure sparse.

    Returns:
        List of Pinecone match dicts.
    """
    dense  = dense_encoder.encode(query, normalize_embeddings=True).tolist()
    dense  = (np.array(dense, dtype=np.float32) * float(alpha)).tolist()
    sparse = bm25_encoder.encode_queries([query])[0]
    sparse = _scale_sparse(sparse, 1.0 - float(alpha))

    res = index.query(
        vector=dense,
        sparse_vector=sparse,
        top_k=int(top_k),
        namespace=namespace,
        include_metadata=True,
        filter=metadata_filter or None,
    )
    matches = res.get("matches") if isinstance(res, dict) else getattr(res, "matches", [])
    return [
        {"id": getattr(m, "id", m.get("id")),
         "score": getattr(m, "score", m.get("score")),
         "metadata": getattr(m, "metadata", m.get("metadata", {}))}
        for m in (matches or [])
    ]


def retrieve_from_namespace(
    index,
    query: str,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    namespace: str,
    top_k: int,
    metadata_filter: Optional[dict] = None,
) -> tuple[list[dict], list[dict]]:
    """
    Retrieve dense and sparse result lists from one namespace.

    Returns:
        (dense_results, sparse_results) — two ranked lists for RRF.
    """
    dense  = _hybrid_query(index, query, dense_encoder, bm25_encoder,
                           alpha=1.0, top_k=top_k, namespace=namespace,
                           metadata_filter=metadata_filter)
    sparse = _hybrid_query(index, query, dense_encoder, bm25_encoder,
                           alpha=0.0, top_k=top_k, namespace=namespace,
                           metadata_filter=metadata_filter)
    return dense, sparse


def reciprocal_rank_fusion(
    ranked_lists: list[list[dict[str, Any]]],
    k: int = 60,
) -> list[dict[str, Any]]:
    """
    Merge multiple ranked lists using Reciprocal Rank Fusion.

    score(d) = sum( 1 / (k + rank_i(d)) )

    Args:
        ranked_lists: each list sorted by relevance descending; items must have 'id'.
        k: RRF constant (default 60).

    Returns:
        Deduplicated list sorted by rrf_score descending.
    """
    scores: dict[str, float] = {}
    best:   dict[str, dict]  = {}
    for lst in ranked_lists:
        for rank, r in enumerate(lst, start=1):
            rid = str(r.get("id", ""))
            if not rid:
                continue
            scores[rid] = scores.get(rid, 0.0) + 1.0 / (float(k) + float(rank))
            if rid not in best:
                best[rid] = r
    fused = [{**best[rid], "rrf_score": sc} for rid, sc in scores.items()]
    fused.sort(key=lambda x: x.get("rrf_score", 0.0), reverse=True)
    return fused


print("Retrieval functions defined.")

Retrieval functions defined.


### Reranking funnel — cross-encoder → MMR → lost-in-middle reorder

Three sequential steps after RRF fusion:
1. **Cross-encoder** (`BAAI/bge-reranker-v2-m3`) scores each (query, passage) pair jointly — more accurate than bi-encoder retrieval. Reduces top-50 → top-10.
2. **MMR** (Maximal Marginal Relevance) — picks the next document that maximises `λ·relevance − (1−λ)·max_similarity_to_selected`. Reduces top-10 → top-5 with diversity.
3. **Lost-in-middle reorder** — places highest-scoring chunk first, second-highest last, rest in middle. Mitigates LLM attention bias.


In [13]:
from __future__ import annotations
from typing import Any, Tuple
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def load_cross_encoder(model_name: str) -> Tuple[Any, Any]:
    """Load cross-encoder tokenizer and model. Returns (tokenizer, model)."""
    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()
    return tok, model


def _get_text(candidate: dict, field: str = "parent_text") -> str:
    """Extract text from a candidate's metadata."""
    md = candidate.get("metadata") or {}
    return str(md.get(field) or md.get("parent_text") or "")


def rerank_cross_encoder(
    query: str,
    candidates: list[dict[str, Any]],
    tokenizer: Any,
    model: Any,
    top_k: int,
) -> list[dict[str, Any]]:
    """
    Score (query, passage) pairs with the cross-encoder and return top_k.

    Args:
        query: user question.
        candidates: RRF-fused results.
        tokenizer, model: loaded cross-encoder.
        top_k: number to keep.

    Returns:
        Top-k candidates sorted by ce_score descending.
    """
    if not candidates:
        return []
    pairs  = [(query, _get_text(c)) for c in candidates]
    inputs = tokenizer(pairs, padding=True, truncation=True,
                       max_length=512, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(-1).tolist()
    if isinstance(logits, float):
        logits = [logits]
    scored = [{**c, "ce_score": float(s)} for c, s in zip(candidates, logits)]
    scored.sort(key=lambda x: x["ce_score"], reverse=True)
    return scored[:top_k]


def _cosine(u: np.ndarray, v: np.ndarray) -> float:
    """Cosine similarity between two vectors."""
    denom = np.linalg.norm(u) * np.linalg.norm(v)
    return float(np.dot(u, v) / denom) if denom > 1e-12 else 0.0


def mmr_filter(
    candidates: list[dict[str, Any]],
    dense_encoder: SentenceTransformer,
    query: str,
    top_k: int,
    lambda_param: float = 0.7,
) -> list[dict[str, Any]]:
    """
    Apply MMR to select top_k diverse candidates.

    Maximises: λ·relevance(d,q) − (1−λ)·max_cosine(d, selected)

    Args:
        candidates: cross-encoder reranked results.
        dense_encoder: used for inter-document similarity.
        query: user question.
        top_k: number to return.
        lambda_param: relevance/diversity trade-off.

    Returns:
        top_k diverse candidates.
    """
    if not candidates:
        return []
    texts   = [_get_text(c) for c in candidates]
    doc_emb = np.asarray(
        dense_encoder.encode(texts, normalize_embeddings=True), dtype=np.float32
    )
    selected:  list[int] = []
    remaining: list[int] = list(range(len(candidates)))

    best0 = int(np.argmax([c.get("ce_score", -1e9) for c in candidates]))
    selected.append(best0)
    remaining.remove(best0)

    while remaining and len(selected) < int(top_k):
        best_idx, best_val = None, -1e18
        for i in remaining:
            rel     = float(candidates[i].get("ce_score", 0.0))
            max_sim = max(_cosine(doc_emb[i], doc_emb[j]) for j in selected)
            score   = lambda_param * rel - (1.0 - lambda_param) * max_sim
            if score > best_val:
                best_val, best_idx = score, i
        if best_idx is None:
            break
        selected.append(best_idx)
        remaining.remove(best_idx)

    return [candidates[i] for i in selected]


def reorder_for_context_window(candidates: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """
    Mitigate lost-in-the-middle bias: place best chunk first, second-best last.

    Args:
        candidates: MMR-filtered results.

    Returns:
        Reordered list.
    """
    if len(candidates) <= 2:
        return list(candidates)
    ordered = sorted(
        candidates,
        key=lambda x: x.get("ce_score", x.get("rrf_score", 0.0)),
        reverse=True,
    )
    return [ordered[0], *ordered[2:], ordered[1]]


ce_tokenizer, ce_model = load_cross_encoder(RERANKER_MODEL)
print("Cross-encoder loaded.")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 16802.53it/s]

Cross-encoder loaded.


### Multi-query retrieval

Gemma 3 generates 3 reformulations of the original query — each targeting a different
aspect of the question (price comparables, trends, amenities). All 4 queries
(original + 3 sub-queries) are run through hybrid retrieval in each relevant namespace.
All result lists are then merged via RRF.


In [14]:
from __future__ import annotations
import re
import ollama


def generate_subqueries(
    query: str,
    n: int = N_SUBQUERIES,
    model: str = OLLAMA_MODEL,
) -> list[str]:
    """
    Generate n alternative query reformulations using Gemma 3.

    Args:
        query: original user question.
        n: number of sub-queries to generate.

    Returns:
        List of n sub-query strings.
    """
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""You are a search query generator for Singapore HDB property data.
Generate {n} alternative search queries that help retrieve relevant data from a vector
database of HDB transactions, amenities, trends, and SHAP features.
Return ONLY a numbered list. No explanations.

Original: {query}
"""
    try:
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
        )
        raw   = (response.get("message") or {}).get("content", "")
        lines = re.findall(r"^\s*\d+\.\s*(.+)$", raw, re.MULTILINE)
        return [l.strip().strip('"') for l in lines[:n]]
    except Exception as e:
        print(f"  Sub-query generation failed: {e}")
        return []


def multi_query_retrieve(
    query: str,
    index,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    namespaces: list[str],
    top_k: int,
    metadata_filter: dict | None = None,
) -> list[dict]:
    """
    Run multi-query hybrid retrieval across all specified namespaces.

    Steps:
    1. Generate N sub-queries.
    2. For each query (original + sub-queries) × each namespace: run dense + sparse.
    3. RRF-fuse all ranked lists.

    Args:
        query: original user question.
        namespaces: list of namespace strings to query.
        metadata_filter: optional Pinecone filter (applied to all namespaces).

    Returns:
        RRF-fused candidate list.
    """
    subqueries  = generate_subqueries(query)
    all_queries = [query] + subqueries

    all_ranked_lists: list[list[dict]] = []

    for q in all_queries:
        for ns in namespaces:
            # Only apply metadata filter on transactions namespace
            filt = metadata_filter if ns == NS_TRANSACTIONS else None
            dense, sparse = retrieve_from_namespace(
                index, q, dense_encoder, bm25_encoder, ns, top_k, filt
            )
            all_ranked_lists.extend([dense, sparse])

    return reciprocal_rank_fusion(all_ranked_lists, k=RRF_K)


print("Multi-query functions defined.")

Multi-query functions defined.


### Full retrieval pipeline composer

Composes every step above into one `retrieve_and_rerank()` function:
1. Extract filters from free-text query via NLP
2. Route to relevant namespaces
3. Multi-query fan-out + hybrid retrieval + RRF
4. Cross-encoder rerank
5. MMR diversity filter
6. Lost-in-middle reorder

This is the function your FastAPI endpoint will call.


In [ ]:
from __future__ import annotations
from typing import Any


def retrieve_and_rerank(
    query: str,
    index,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    ce_tokenizer: Any,
    ce_model: Any,
) -> list[dict]:
    """
    Full RAG retrieval pipeline for a free-text user query.

    Steps:
    1. NLP filter extraction (no metadata required from user)
    2. Namespace routing
    3. Multi-query hybrid retrieval + RRF
    4. Cross-encoder rerank (top-50 → top-10)
    5. MMR diversity (top-10 → top-5)
    6. Lost-in-middle reorder

    Args:
        query: raw free-text question from the user.

    Returns:
        top-5 reordered context chunks.
    """
    # 1. Extract filters
    metadata_filter = extract_filters_from_query(query)

    # 2. Route namespaces
    namespaces = route_namespaces(query)

    # 3. Multi-query + hybrid + RRF
    fused = multi_query_retrieve(
        query=query,
        index=index,
        dense_encoder=dense_encoder,
        bm25_encoder=bm25_encoder,
        namespaces=namespaces,
        top_k=TOP_K_RETRIEVAL,
        metadata_filter=metadata_filter,
    )

    # 4. Cross-encoder rerank
    reranked = rerank_cross_encoder(
        query=query,
        candidates=fused[:TOP_K_RETRIEVAL],
        tokenizer=ce_tokenizer,
        model=ce_model,
        top_k=TOP_K_RERANK,
    )

    # 5. MMR
    diverse = mmr_filter(
        candidates=reranked,
        dense_encoder=dense_encoder,
        query=query,
        top_k=TOP_K_MMR,
        lambda_param=MMR_LAMBDA,
    )

    # 6. Reorder
    return reorder_for_context_window(diverse)[:TOP_K_FINAL]


# ── smoke test ─────────────────────────────────────────────────────────────────
smoke_query = "Is $580k fair for a 4-room flat in Tampines?"
context     = retrieve_and_rerank(
    query=smoke_query,
    index=index,
    dense_encoder=dense_encoder,
    bm25_encoder=bm25_encoder,
    ce_tokenizer=ce_tokenizer,
    ce_model=ce_model,
)
print(f"Retrieved {len(context)} context chunks")
for i, c in enumerate(context, 1):
    m = c.get("metadata") or {}
    print(f"  [{i}] source={m.get('source')} | town={m.get('town')} | year={m.get('sale_year')}")

Retrieved 5 context chunks
  [1] source=transaction | town=TAMPINES | year=2018
  [2] source=transaction | town=TAMPINES | year=2018
  [3] source=transaction | town=TAMPINES | year=2018
  [4] source=transaction | town=TAMPINES | year=2022
  [5] source=transaction | town=TAMPINES | year=2018


: 

### Prediction tool — joblib model call from free-text query

This is a new tool that extracts a structured `PredictRequest`-shaped dict from the
user's free-text query, then calls the PropertyLens `HybridClusterEnsemble` model
directly via joblib.

The prediction result (price estimate + confidence band) is returned as a formatted
string that gets appended to the LLM prompt alongside the RAG context.

**Entity extraction uses Gemma 3** to parse fields like `floor_area_sqm`, `storey_range`,
`town`, `flat_type`, `lease_commence_date` from the query.
If a required field cannot be extracted the tool returns a fallback message.


In [ ]:
from __future__ import annotations
import json
import re
import joblib
import pandas as pd
import ollama


def load_model_bundle(path: str) -> dict | None:
    """
    Load the PropertyLens joblib model bundle.

    Args:
        path: path to model_bundle.pkl.

    Returns:
        dict bundle or None if not found.
    """
    import os
    if not os.path.exists(path):
        print(f"  Model bundle not found at {path} — prediction tool disabled.")
        return None
    bundle = joblib.load(path)
    print(f"  Model bundle loaded. Keys: {list(bundle.keys())[:8]}")
    return bundle


def extract_predict_request(
    query: str,
    model: str = OLLAMA_MODEL,
) -> dict | None:
    """
    Use Gemma 3 to extract a PredictRequest-shaped dict from a free-text query.

    Target fields (matching PropertyLens PredictRequest schema):
        town, flat_type, floor_area_sqm, storey_range,
        lease_commence_date, flat_model (optional)

    Returns None if insufficient fields extracted.

    Args:
        query: raw user question.

    Returns:
        dict or None.
    """
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""Extract fields for a Singapore HDB price prediction from this query.
Return ONLY a JSON object with these keys (omit if not mentioned):
  town             : string, ALL CAPS HDB town e.g. "TAMPINES"
  flat_type        : string, e.g. "4 ROOM", "5 ROOM", "3 ROOM"
  floor_area_sqm   : number, floor area in square metres
  storey_range     : string, e.g. "07 TO 09" or just "high"/"mid"/"low"
  lease_commence_date: integer year the flat lease started e.g. 1990
  flat_model       : string, e.g. "New Generation", "Improved"
Return {{}} if nothing is clear. Return ONLY JSON.

Query: {query}
"""
    try:
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
        )
        raw    = (response.get("message") or {}).get("content", "{}")
        raw    = re.sub(r"```[\w]*", "", raw).strip()
        parsed = json.loads(raw)
        # Require at minimum town + flat_type
        if parsed.get("town") and parsed.get("flat_type"):
            return parsed
        return None
    except Exception as e:
        print(f"  PredictRequest extraction failed: {e}")
        return None


def run_prediction(
    predict_request: dict,
    model_bundle: dict,
) -> str:
    """
    Call the HybridClusterEnsemble model and return a formatted prediction string.

    Looks for a 'predict' callable or a 'model' key in the bundle.

    Args:
        predict_request: dict with HDB flat features.
        model_bundle: loaded joblib bundle dict.

    Returns:
        Formatted prediction result string, or error message.
    """
    model_obj = model_bundle.get("model") or model_bundle.get("ensemble")
    if model_obj is None:
        return "[Prediction tool: no model object found in bundle.]"

    try:
        row = pd.DataFrame([predict_request])
        prediction = model_obj.predict(row)
        price = float(prediction[0]) if hasattr(prediction, "__len__") else float(prediction)

        # Attempt confidence band (±10% as fallback if not provided)
        low  = price * 0.90
        high = price * 1.10

        return (
            f"Model price estimate: SGD {int(round(price)):,} "
            f"(confidence band: SGD {int(round(low)):,} – SGD {int(round(high)):,}). "
            f"Input used: {predict_request}"
        )
    except Exception as e:
        return f"[Prediction tool error: {type(e).__name__}: {e}]"


def prediction_tool(query: str, model_bundle: dict | None) -> str:
    """
    End-to-end prediction tool: free-text query → price estimate string.

    Returns a formatted string ready to be appended to the LLM prompt.
    Returns an empty string if the model bundle is unavailable.

    Args:
        query: raw user question.
        model_bundle: loaded bundle or None.

    Returns:
        Prediction result string.
    """
    if model_bundle is None:
        return ""
    predict_request = extract_predict_request(query)
    if predict_request is None:
        return "[Prediction tool: insufficient fields extracted from query — skipping prediction.]"
    return run_prediction(predict_request, model_bundle)


# ── load model bundle ──────────────────────────────────────────────────────────
model_bundle = load_model_bundle(MODEL_BUNDLE_PATH)

# ── smoke test ─────────────────────────────────────────────────────────────────
test_q  = "Is $580k fair for a 4-room HDB in Tampines, 100 sqm, mid-floor?"
pred_result = prediction_tool(test_q, model_bundle)
print("Prediction tool result:")
print(pred_result)

### Prompt builder and Gemma 3 answer generation

The prompt injects:
1. Retrieved context chunks (labelled `[Context N]`)
2. The prediction tool result (if available)
3. The user's original question

The system prompt enforces strict grounding — Gemma 3 must cite context labels and
cannot use outside knowledge.


In [ ]:
from __future__ import annotations
from typing import Any
import ollama


def build_system_prompt() -> str:
    """Return the system prompt for Gemma 3."""
    return """You are a Singapore HDB property pricing assistant for the PropertyLens app.
Help buyers and sellers make informed decisions about HDB resale prices.

Rules:
1. Answer ONLY using the provided context and model prediction (if given). No outside knowledge.
2. Cite each claim with [Context N] labels.
3. If a model prediction is provided, reference it explicitly in your answer.
4. Give a clear verdict: Fair / Above market / Below market.
5. If evidence is thin, say so. Keep answers to 3-5 sentences unless detail is requested.
"""


def build_rag_prompt(
    query: str,
    context_chunks: list[dict[str, Any]],
    prediction_result: str = "",
) -> str:
    """
    Build the full user-turn prompt string.

    Args:
        query: user question.
        context_chunks: retrieved + reranked chunks.
        prediction_result: output of prediction_tool() — empty string to omit.

    Returns:
        Formatted prompt string.
    """
    parts = ["## Retrieved context"]
    for i, c in enumerate(context_chunks, 1):
        md  = c.get("metadata") or {}
        txt = str(md.get("parent_text") or "").strip()
        hdr = (
            f"[Context {i}] source={md.get('source')} "
            f"town={md.get('town')} year={md.get('sale_year')}"
        )
        parts.append(hdr)
        parts.append(txt)
        parts.append("")

    if prediction_result:
        parts.append("## Model prediction")
        parts.append(prediction_result)
        parts.append("")

    parts.append("## Question")
    parts.append(query)
    return "\n".join(parts).strip()


def generate_answer(
    query: str,
    context_chunks: list[dict[str, Any]],
    prediction_result: str = "",
    model: str = OLLAMA_MODEL,
) -> str:
    """
    Generate a grounded answer using Ollama + Gemma 3.

    Args:
        query: original user question.
        context_chunks: final reordered context from retrieve_and_rerank().
        prediction_result: optional model prediction string.
        model: Ollama model name.

    Returns:
        Answer string.
    """
    _set_ollama_host(OLLAMA_BASE_URL)
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": build_system_prompt()},
                {"role": "user",   "content": build_rag_prompt(query, context_chunks, prediction_result)},
            ],
        )
        return (response.get("message") or {}).get("content", "")
    except Exception as e:
        return f"[Ollama error] {type(e).__name__}: {e}"


print("Prompt builder and generate_answer defined.")

### End-to-end demo — free-text queries, no metadata required

Five demo queries covering different personas and namespaces.
The user only types a natural language question — no `metadata_filter` dict needed.
Filter extraction, namespace routing, retrieval, reranking, prediction, and generation
all happen automatically inside `run_demo()`.


In [ ]:
from __future__ import annotations

DEMO_QUERIES = [
    {
        "query":   "Is $580k fair for a 4-room HDB in Tampines Street 81, Block 432?",
        "persona": "Buyer",
    },
    {
        "query":   "What should I list my 5-room Bishan flat for given current market trends?",
        "persona": "Seller",
    },
    {
        "query":   "Are HDB prices in Queenstown rising or falling over the last 3 years?",
        "persona": "Trends",
    },
    {
        "query":   "What amenities are near Bedok North? Any MRT stations or schools?",
        "persona": "Amenities",
    },
    {
        "query":   "Why did the model predict a high price for this Queenstown flat? What features drove it?",
        "persona": "XAI",
    },
]


def run_demo(demo: dict) -> None:
    """Run one demo query end-to-end and print a formatted result."""
    query = demo["query"]
    print(f"\n{'='*60}")
    print(f"[{demo['persona']}] {query}")
    print(f"{'='*60}")

    # ── step 1: show extracted filter + namespaces ─────────────────────────────
    metadata_filter = extract_filters_from_query(query)
    namespaces      = route_namespaces(query)
    print(f"  Extracted filter : {metadata_filter}")
    print(f"  Namespaces       : {namespaces}")

    # ── step 2: retrieve + rerank ──────────────────────────────────────────────
    ctx = retrieve_and_rerank(
        query=query,
        index=index,
        dense_encoder=dense_encoder,
        bm25_encoder=bm25_encoder,
        ce_tokenizer=ce_tokenizer,
        ce_model=ce_model,
    )
    print(f"\n  Context chunks ({len(ctx)}):")
    for i, c in enumerate(ctx, 1):
        m = c.get("metadata") or {}
        source = m.get("source", "")

        if source == "transaction":
            rp = m.get("resale_price")
            rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
            print(f"    [{i}] transaction | {m.get('town')} | {m.get('flat_type')} | {rp_str} | {m.get('sale_year')}")

        elif source == "amenity":
            print(f"    [{i}] amenity | {m.get('town')} | {m.get('amenity_type')} | count={m.get('count')}")

        elif source == "trend":
            rp = m.get("resale_price")
            rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
            print(f"    [{i}] trend | {m.get('town')} | median={rp_str} | {m.get('sale_year')}")

        elif source == "xai":
            print(f"    [{i}] xai | type={m.get('xai_type')} | {m.get('parent_text', '')[:80]}...")

        else:
            print(f"    [{i}] {source} | {m}")
    # ── step 3: prediction tool ────────────────────────────────────────────────
    pred = prediction_tool(query, model_bundle)
    if pred:
        print(f"\n  Prediction : {pred[:120]}..." if len(pred) > 120 else f"\n  Prediction : {pred}")

    # ── step 4: generate answer ────────────────────────────────────────────────
    answer = generate_answer(query, ctx, pred)
    print(f"\n  Answer:\n{answer}")


for demo in DEMO_QUERIES:
    run_demo(demo)

### Known limitations and next steps

#### Known limitations
- School quality not included — amenities contain proximity only, not ranking/popularity.
- Transaction data may only cover specific years depending on which CSVs were ingested — check `transactions_df.transaction_year.value_counts()`.
- `extract_predict_request()` requires Gemma 3 to reliably extract `floor_area_sqm` — queries that omit this will fall back to no prediction.
- Pinecone free tier has index size limits — use paid tier for full transaction history.
- Gemma 3 (local) is smaller than GPT-4 class models; complex multi-hop reasoning may be weaker.

#### Suggested next steps
- Expose `retrieve_and_rerank()` + `prediction_tool()` as `/api/rag/query` in `backend/main.py`.
- Evaluate retrieval quality with a labelled test set using NDCG@5.
- Add MOE school popularity data (Phase 2A oversubscription) to amenities CSVs.
- Fine-tune BGE-M3 on PropertyLens domain queries to improve retrieval precision.
- Replace `extract_predict_request()` NLP extraction with a small structured-output call using the Anthropic API for higher reliability.


In [ ]:
# PropertyGuru listing: 1 Lorong Lew Lian, Serangoon
# Asking price: $430,000 | 689 sqft (64.0 sqm) | 3I HDB | TOP 1977 | 52yr lease remaining

test_query = (
    "Is $430,000 fair for a 3-room HDB flat at 1 Lorong Lew Lian, Serangoon? "
    "The flat is 689 sqft (64 sqm), mid-floor, lease started 1978, "
    "52 years remaining, 3 mins walk to Serangoon MRT."
)

# ── step 1: extract structured fields ─────────────────────────────────────────
predict_request = extract_predict_request(test_query)
print("Extracted PredictRequest:")
print(predict_request)

# ── step 2: run prediction ─────────────────────────────────────────────────────
pred = prediction_tool(test_query, model_bundle)
print("\nPrediction tool result:")
print(pred)

# ── step 3: full RAG + answer ──────────────────────────────────────────────────
ctx = retrieve_and_rerank(
    query=test_query,
    index=index,
    dense_encoder=dense_encoder,
    bm25_encoder=bm25_encoder,
    ce_tokenizer=ce_tokenizer,
    ce_model=ce_model,
)

print(f"\nContext chunks ({len(ctx)}):")
for i, c in enumerate(ctx, 1):
    m = c.get("metadata") or {}
    source = m.get("source", "")
    if source == "transaction":
        rp = m.get("resale_price")
        rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
        print(f"  [{i}] transaction | {m.get('town')} | {m.get('flat_type')} | {rp_str} | {m.get('sale_year')}")
    elif source == "amenity":
        print(f"  [{i}] amenity | {m.get('town')} | {m.get('amenity_type')} | count={m.get('count')}")
    elif source == "trend":
        rp = m.get("resale_price")
        rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
        print(f"  [{i}] trend | {m.get('town')} | median={rp_str} | {m.get('sale_year')}")
    elif source == "xai":
        print(f"  [{i}] xai | type={m.get('xai_type')} | {m.get('parent_text', '')[:80]}...")

answer = generate_answer(test_query, ctx, pred)
print("\n" + "="*60)
print("Query: Is $430k fair for 1 Lorong Lew Lian, Serangoon?")
print("="*60)
print(answer)

In [ ]:
print(list(model_bundle.keys()))